In [1]:
# %pip install -U kaleido
# %pip install -U plotly
# !plotly_get_chrome
# %pip install -U nbformat jupyter ipywidgets

In [2]:
import os
import pandas as pd

from plotly import graph_objects as go
from plotly.io import write_image

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.13f' % x)

In [3]:
UEA_MTSC30 = ['EthanolConcentration',
              'FaceDetection',
              'Handwriting',
              'Heartbeat',
              'JapaneseVowels',
              'PEMS-SF',
              'SelfRegulationSCP1',
              'SelfRegulationSCP2',
              'SpokenArabicDigits',
              'UWaveGestureLibrary',
              'ArticularyWordRecognition',
              'AtrialFibrillation',
              'BasicMotions',
              'CharacterTrajectories',
              'Cricket',
              'DuckDuckGeese',
              'EigenWorms',
              'Epilepsy',
              'ERing',
              'FingerMovements',
              'HandMovementDirection',
              'InsectWingbeat',
              'Libras',
              'LSST',
              'MotorImagery',
              'NATOPS',
              'PenDigits',
              'PhonemeSpectra',
              'RacketSports',
              'StandWalkJump']

In [8]:
best_result = []

for data_name in (UEA_MTSC30):
    try:
        with open(f'../03-full_results/MambaSL/_proposed/MambaSL_CLS_{data_name}_proposed.out', 'r') as file:
            data = file.read().splitlines()
    except:
        print(f'File not found for {data_name}, skipping...')
        continue
    
    result_lst = list()
    for i in range(len(data)):
        if data[i].startswith('>>>>>>>testing : '):
            data_meta = list(data[i][17:-33].split('_'))
            data_meta[6] = int(data_meta[6].replace('sl', ''))
            data_meta[7] = int(data_meta[7].replace('ll', ''))
            data_meta[8] = int(data_meta[8].replace('pl', ''))
            data_meta[9] = int(data_meta[9].replace('dm', ''))
            data_meta[10] = int(data_meta[10].replace('ds', ''))
            data_meta[11] = int(data_meta[11].replace('expand', ''))
            data_meta[12] = int(data_meta[12].replace('dc', ''))
            data_meta[13] = int(data_meta[13].replace('nk', ''))
            data_meta[14] = bool(int(data_meta[14].replace('tvdt', '')))
            data_meta[15] = bool(int(data_meta[15].replace('tvB', '')))
            data_meta[16] = bool(int(data_meta[16].replace('tvC', '')))
            data_meta[17] = bool(int(data_meta[17].replace('useD', '')))
            if data[i+2].startswith('gating_value'):
                acc = float(data[i+3].replace('accuracy:', ''))
                model_params = data[i+4].replace('model parameter : ', '')
                model_size = data[i+5].replace('model size : ', '').replace('MB', '')
            else:
                acc = float(data[i+2].replace('accuracy:', ''))
                model_params = data[i+3].replace('model parameter : ', '')
                model_size = data[i+4].replace('model size : ', '').replace('MB', '')

            result_data = {
                # 'task': data_meta[0],
                # 'model_id': data_meta[1],
                'data_name': data_meta[2],
                'model': data_meta[3],
                # 'data': data_meta[4],
                # 'feature': data_meta[5],
                'seq_len': data_meta[6],
                # 'label_len': data_meta[7],
                # 'pred_len': data_meta[8],
                'd_model': data_meta[9],
                'd_state': data_meta[10],
                'expand': data_meta[11],
                'd_conv': data_meta[12],
                'num_kernel': data_meta[13],
                'tv_dt': data_meta[14],
                'tv_B': data_meta[15],
                'tv_C': data_meta[16],
                'use_D': data_meta[17],
                'exp': data_meta[18],
                # 'desc': data_meta[19],
                'acc': acc,
                'model_params': int(model_params),
                'model_size (MB)': float(model_size),
                'ckpt': data[i].replace('>>>>>>>testing : ', '').replace('<', '')
            }

            result_lst.append(result_data)

    result_df = pd.DataFrame(result_lst)

    print(data_name, len(result_df))
    print('best result', result_df['acc'].max())
    # display(result_df[result_df['acc'] == result_df['acc'].max()].sort_values('model_params'))

    emoji = {
        True: 'O',
        False: 'X'
    }
    tmp_result = result_df.pivot_table(index=['d_model', 'd_state', 'expand'], columns=['tv_dt','tv_B','tv_C',], values='acc')
    tmp_result = pd.DataFrame(tmp_result.values, 
                              columns=[f'TV-Δ:{emoji[bool(tv_dt)]}\nTV-B:{emoji[bool(tv_B)]}\nTV-C:{emoji[bool(tv_C)]}' 
                                       for tv_dt, tv_B, tv_C in tmp_result.columns],
    )

    # select top lines
    tmp_result = tmp_result.loc[tmp_result.mean(axis=1).nlargest(10).index]

    # reorder columns by max -> mean
    tmp_result = tmp_result.reindex(tmp_result.mean(axis=0).sort_values(ascending=False).index, axis=1)
    tmp_result = tmp_result.reindex(tmp_result.max(axis=0).sort_values(ascending=False).index, axis=1)
    
    tmp_result_box = tmp_result.melt(var_name='Configuration', value_name='Accuracy')

    fig = go.Figure()
    fig.add_trace(go.Box(x=tmp_result_box['Configuration'], y=tmp_result_box['Accuracy']))
    for idx in tmp_result.index:
        fig.add_trace(go.Scatter(x=tmp_result.columns,
                                 y=tmp_result.loc[idx], mode='lines+markers', name=idx, line=dict(width=1), marker=dict(size=6, opacity=0.5)))

    

    fig.update_layout(xaxis_title='configuration', yaxis_title='average accuracy', showlegend=False)
    fig.update_layout(margin=dict(l=5, r=5, t=5, b=20),
                      width=600, height=250)
    fig.update_xaxes(
        tickangle=0,
        tickvals=tmp_result.columns,
        ticktext=[col.replace('\n', '<br>').replace(
            'X', "<span style='color:#BC2023'>✗</span>").replace(
                'O', "<span style='color:#0C6B37'>✓</span>") for col in tmp_result.columns],
        tickfont=dict(size=15,
                      family='Menlo, sans-serif'),
        title_standoff=10
    )
    fig.update_yaxes(title_standoff=5)
    display(fig)
    os.makedirs('figures_tmp', exist_ok=True)
    # write_image(fig, f'figures_tmp/{data_name}.pdf', width=600, height=250, scale=2, format='pdf')

    best_result.append(tmp_result.T.max(axis=1).to_dict())


EthanolConcentration 240
best result 0.42585551330798477


FaceDetection 240
best result 0.6929625425652668


Handwriting 240
best result 0.6082352941176471


Heartbeat 240
best result 0.8048780487804879


JapaneseVowels 240
best result 0.9864864864864865


...

NATOPS 240
best result 0.9888888888888889


PenDigits 240
best result 0.9925671812464265


PhonemeSpectra 240
best result 0.3033104682373993


RacketSports 240
best result 0.9276315789473685


StandWalkJump 240
best result 0.7333333333333333


In [9]:
best_results = pd.DataFrame(best_result)
best_results.index = UEA_MTSC30
best_results * 100

,TV-Δ:X\nTV-B:X\nTV-C:X,TV-Δ:X\nTV-B:X\nTV-C:O,TV-Δ:X\nTV-B:O\nTV-C:O,TV-Δ:X\nTV-B:O\nTV-C:X,TV-Δ:O\nTV-B:X\nTV-C:X,TV-Δ:O\nTV-B:X\nTV-C:O,TV-Δ:O\nTV-B:O\nTV-C:X,TV-Δ:O\nTV-B:O\nTV-C:O
EthanolConcentration,42.5855513307985,39.5437262357414,36.5019011406844,35.7414448669202,33.4600760456274,33.0798479087452,32.6996197718631,30.7984790874525
FaceDetection,68.5868331441544,68.3881952326901,68.8422247446084,68.8422247446084,68.0760499432463,69.2962542565267,68.4449489216799,69.0124858115777
Handwriting,59.2941176470588,59.6470588235294,58.5882352941177,56.4705882352941,60.1176470588235,60.8235294117647,56.9411764705882,54.1176470588235
Heartbeat,80.4878048780488,79.5121951219512,79.0243902439024,79.0243902439024,80.0000000000000,79.5121951219512,79.0243902439024,79.5121951219512
JapaneseVowels,98.3783783783784,98.3783783783784,98.1081081081081,98.3783783783784,98.1081081081081,98.3783783783784,98.6486486486486,98.1081081081081
PEMS-SF,81.5028901734104,82.0809248554913,83.2369942196532,85.5491329479769,82.6589595375723,80.3468208092486,85.5491329479769,81.5028901734104
SelfRegulationSCP1,91.4675767918089,90.7849829351536,90.1023890784983,90.1023890784983,90.1023890784983,90.7849829351536,91.4675767918089,90.4436860068259
SelfRegulationSCP2,57.7777777777778,58.3333333333333,58.8888888888889,58.8888888888889,57.7777777777778,57.7777777777778,57.7777777777778,65.0000000000000
SpokenArabicDigits,99.5907230559345,99.5907230559345,99.6816734879491,99.9545247839927,99.7271487039563,99.8635743519782,99.7271487039563,99.6361982719418
UWaveGestureLibrary,93.1250000000000,93.1250000000000,91.2500000000000,91.5625000000000,92.5000000000000,93.1250000000000,92.8125000000000,91.2500000000000


In [10]:
best_results.mean()

TV-Δ:X\nTV-B:X\nTV-C:X   0.7740007950556
TV-Δ:X\nTV-B:X\nTV-C:O   0.7701225369550
TV-Δ:X\nTV-B:O\nTV-C:O   0.7776716014442
TV-Δ:X\nTV-B:O\nTV-C:X   0.7704175818896
TV-Δ:O\nTV-B:X\nTV-C:X   0.7770721618753
TV-Δ:O\nTV-B:X\nTV-C:O   0.7714841525733
TV-Δ:O\nTV-B:O\nTV-C:X   0.7708467975093
TV-Δ:O\nTV-B:O\nTV-C:O   0.7705455970654
dtype: float64